# Tech Challenge — Fase 3
## Assistente Virtual Médico — Hospital Vida Plena (fictício)

**Pós-graduação em IA para Devs — FIAP**

Este notebook é o ponto único de execução do projeto para o Google Colab. Ele:

1. Clona o repositório do projeto e instala as dependências;
2. Prepara e anonimiza o dataset de fine-tuning (protocolos internos + amostra estilo MedQuAD);
3. Realiza o fine-tuning (LoRA) de um LLM com os dados médicos internos;
4. Avalia o modelo fine-tuned comparando com o modelo base (ROUGE-L);
5. Indexa os protocolos internos em um banco vetorial (RAG com LangChain);
6. Executa o fluxo automatizado do assistente médico (LangGraph): verifica exames/alertas, consulta o RAG, gera a resposta com o LLM e aplica os guardrails de segurança;
7. Mostra os logs de auditoria de cada interação.

> **Aviso**: todos os dados clínicos usados aqui (protocolos, prontuários, pacientes) são **fictícios**, criados apenas para fins didáticos deste Tech Challenge. Nenhum dado real de paciente foi utilizado.

> **Como rodar**: em Colab, use um runtime com GPU (Menu *Ambiente de execução* → *Alterar tipo de ambiente de execução* → GPU T4). O notebook também roda em CPU/MPS local, apenas mais lentamente.


## 0. Setup — clonar repositório e instalar dependências

In [ ]:
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules
print("Rodando no Google Colab:", IN_COLAB)

# --- Preencha com a URL do seu repositório Git antes de rodar no Colab ---
REPO_URL = "https://github.com/SEU_USUARIO/tech-challenge-fase3-assistente-medico.git"
REPO_DIR = "tech-challenge-fase3-assistente-medico"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    # Execução local: assume que o notebook está em notebooks/ dentro do repo
    os.chdir(os.path.dirname(os.path.abspath(".")))

print("Diretório de trabalho:", os.getcwd())
sys.path.insert(0, os.getcwd())


In [ ]:
%pip install -q -r requirements.txt
# O Colab costuma vir com torchao desatualizado, o que quebra o carregamento
# de modelos no transformers recente (ImportError: incompatible version of torchao).
%pip install -q -U torchao


> **Se aparecer `ImportError: Found an incompatible version of torchao`**: rode a célula acima novamente e, em seguida, reinicie o ambiente de execução (*Ambiente de execução → Reiniciar sessão*) antes de continuar — o Colab mantém em memória a versão antiga do pacote já importada. Depois de reiniciar, **não** é necessário clonar o repositório de novo, apenas rode a célula 0 novamente.


In [ ]:
import torch
print("CUDA disponível:", torch.cuda.is_available())
print("MPS disponível:", torch.backends.mps.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Preparação e anonimização dos dados

Combina uma amostra no estilo MedQuAD (ou o MedQuAD real, se baixado em `data/raw/MedQuAD`) com perguntas frequentes extraídas dos protocolos internos do Hospital Vida Plena, aplicando curadoria (dedup, filtro de tamanho) e anonimização de PII antes de qualquer uso em treinamento.

Para usar o MedQuAD real (milhares de pares de pergunta/resposta clínicas), descomente a célula abaixo — o download é feito diretamente do repositório oficial.

In [ ]:
# Opcional: baixar o dataset MedQuAD real (milhares de QA clínicos)
# subprocess.run(["git", "clone", "https://github.com/abachaa/MedQuAD.git", "data/raw/MedQuAD"], check=True)


In [ ]:
from src.preprocessing.anonymize import anonymize_text
from src.preprocessing.prepare_dataset import build_finetuning_dataset

# Demonstração da anonimização em um documento de exemplo com PII fictício
with open("data/raw/exemplo_com_pii.txt", encoding="utf-8") as f:
    texto_original = f.read()

print("--- ANTES ---")
print(texto_original)
print("--- DEPOIS (anonimizado) ---")
print(anonymize_text(texto_original))


In [ ]:
dataset_path = build_finetuning_dataset()

import json
with open(dataset_path, encoding="utf-8") as f:
    registros = [json.loads(l) for l in f]

print(f"Total de exemplos curados: {len(registros)}")
for r in registros[:3]:
    print("-", r["instruction"], "->", r["response"][:80], "...")


## 2. Fine-tuning do LLM (LoRA) com dados médicos internos

Usa o modelo aberto `TinyLlama/TinyLlama-1.1B-Chat-v1.0` como base e aplica LoRA (Parameter-Efficient Fine-Tuning) sobre o dataset preparado na etapa anterior. Em GPU T4 (Colab), o treinamento completo leva poucos minutos; em CPU/MPS é mais lento.

Para um cenário real de hospital, o dataset seria muito maior (milhares de protocolos, laudos e perguntas reais anonimizadas) e o modelo base poderia ser trocado por um LLM maior (Llama 3, Falcon etc.).

In [ ]:
from src.finetuning.train import main as train_main
import sys as _sys

# Executa o fine-tuning via linha de comando simulada (reaproveita o script de produção)
_sys.argv = [
    "train.py",
    "--model", "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "--epochs", "3",
    "--batch-size", "2",
]
train_main()


## 3. Avaliação do modelo fine-tuned vs. modelo base

In [ ]:
from src.finetuning.evaluate import evaluate
import pandas as pd

report = evaluate(
    base_model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    adapter_dir="models/llm-medico-lora",
    dataset_path="data/processed/finetuning_dataset.jsonl",
    n_samples=5,
)

print(f"ROUGE-L médio — base: {report['avg_rouge_l_base']:.4f} | fine-tuned: {report['avg_rouge_l_finetuned']:.4f}")
pd.DataFrame(report["samples"])[["instruction", "base_model_answer", "finetuned_model_answer", "rouge_l_base", "rouge_l_finetuned"]]


## 4. Indexação RAG dos protocolos internos (LangChain + FAISS)

Cada protocolo é dividido em chunks, transformado em embeddings (modelo local `sentence-transformers/all-MiniLM-L6-v2`) e indexado em um banco vetorial FAISS. Cada chunk mantém o nome do arquivo de origem como metadado, permitindo citar a fonte na resposta final (explainability).

In [ ]:
from src.rag.indexing import build_index

build_index()


## 5. Fluxo automatizado do assistente médico (LangGraph)

O grafo abaixo coordena as etapas exigidas pelo desafio:

`carregar_prontuario` → `verificar_exames_e_alertas` → `consultar_rag` → `gerar_resposta` → `aplicar_guardrails` → `registrar_auditoria`

A geração de texto usa o modelo fine-tuned carregado como um `transformers.pipeline`, injetado no grafo via `make_hf_pipeline_generator`.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftModel
from src.agent.llm_client import make_hf_pipeline_generator
from src.agent.graph import run_assistente

BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_DIR = "models/llm-medico-lora"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
finetuned_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

gerador_pipe = pipeline("text-generation", model=finetuned_model, tokenizer=tokenizer)
generator = make_hf_pipeline_generator(gerador_pipe)


In [ ]:
# Exemplo 1 — paciente com critérios de sepse (qSOFA alto)
resultado = run_assistente(
    pergunta="Quais cuidados iniciais para esse paciente?",
    paciente_id="P-0001",
    generator=generator,
)

print("ALERTAS ATIVOS:")
for a in resultado["alertas"]:
    print("-", a)

print("\nFONTES CITADAS:", [c["source"] for c in resultado["contexto_protocolo"]])
print("\nRESPOSTA FINAL:\n", resultado["resposta_final"])
print("\nGUARDRAIL FLAGS:", resultado["guardrail_flags"])


In [ ]:
# Exemplo 2 — paciente em crise hipertensiva
resultado_2 = run_assistente(
    pergunta="Como devo conduzir esse caso de pressão alta?",
    paciente_id="P-0002",
    generator=generator,
)
print("ALERTAS:", resultado_2["alertas"])
print("\nRESPOSTA FINAL:\n", resultado_2["resposta_final"])


In [ ]:
# Exemplo 3 — paciente com suspeita de dengue e sinal de alarme
resultado_3 = run_assistente(
    pergunta="Esse paciente precisa ser internado?",
    paciente_id="P-0003",
    generator=generator,
)
print("ALERTAS:", resultado_3["alertas"])
print("\nRESPOSTA FINAL:\n", resultado_3["resposta_final"])


## 6. Guardrail em ação — bloqueio de prescrição direta

Demonstração isolada do guardrail de segurança: mesmo que o LLM gere uma resposta com linguagem de prescrição direta, o sistema sinaliza a flag `linguagem_de_prescricao_direta_detectada` e garante o disclaimer de validação humana.

In [ ]:
from src.security.guardrails import check_response

resposta_arriscada = "Tome 500mg de dipirona a cada 6 horas."
resultado_guardrail = check_response(resposta_arriscada, has_sources=False)

print("Bloqueado:", resultado_guardrail["blocked"])
print("Flags:", resultado_guardrail["flags"])
print("Resposta segura entregue ao médico:\n", resultado_guardrail["safe_response"])


## 7. Logs de auditoria

In [ ]:
from src.security.audit_log import read_audit_log
import pandas as pd

logs = read_audit_log()
pd.json_normalize(logs)


## Conclusão

Este notebook demonstrou o ciclo completo do assistente virtual médico do Hospital Vida Plena: fine-tuning de um LLM com dados internos anonimizados, RAG com LangChain sobre os protocolos do hospital, orquestração do fluxo de decisão com LangGraph (verificação de exames/alertas, geração de resposta contextualizada, guardrails de segurança) e auditoria completa via logging estruturado.

Para uma implantação real, os próximos passos seriam: (1) substituir os dados fictícios por dados reais do hospital, devidamente anonimizados e aprovados pelo DPO/compliance; (2) treinar com um dataset muito maior; (3) validar clinicamente as respostas com a equipe médica antes de qualquer uso assistencial.